# AquaCast Model Pipeline Version 10

This notebook trains the current best backend models for AquaCast / BeachGuard using the weekly enhanced dataset.

**Dataset path used in Colab:**

```python
/content/drive/MyDrive/Datasets/Aquacast15Years_Weekly.csv
```

Current selected models:
- **E. coli:** Logistic Regression using expanded no-tides features
- **Enterococcus:** Logistic Regression using base no-SSO features

This version saves the trained model files, feature lists, thresholds, metrics, prediction CSVs, confusion-matrix images, and a downloadable ZIP file, similar to the earlier V4/V5 notebooks.


In [ ]:
# ============================================================
# 1. Setup
# ============================================================

import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    average_precision_score,
    roc_auc_score,
)

RANDOM_STATE = 42

DATA_PATH = "/content/drive/MyDrive/Datasets/Aquacast15Years_Weekly.csv"

LOCAL_FALLBACK_PATHS = [
    "/mnt/data/Aquacast15Years_enhanced_no_tides.csv",
    "/mnt/data/Aquacast15Years_Weekly.csv",
]

In [ ]:
# ============================================================
# 2. Mount Drive and load dataset
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Not running in Colab or Drive already mounted.")

if Path(DATA_PATH).exists():
    data_file = DATA_PATH
else:
    data_file = None
    for p in LOCAL_FALLBACK_PATHS:
        if Path(p).exists():
            data_file = p
            break

if data_file is None:
    raise FileNotFoundError(
        f"Could not find dataset at {DATA_PATH}. "
        "Upload Aquacast15Years_Weekly.csv to MyDrive/Datasets."
    )

df = pd.read_csv(data_file)
print("Loaded:", data_file)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# ============================================================
# 3. Standardize column names
# ============================================================

def normalize_col(c):
    return (
        str(c)
        .strip()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace(".", "")
        .lower()
    )

df.columns = [normalize_col(c) for c in df.columns]

rename_map = {}
if "sampledate" in df.columns and "sample_date" not in df.columns:
    rename_map["sampledate"] = "sample_date"
if "indicator" in df.columns and "indicator_clean" not in df.columns:
    rename_map["indicator"] = "indicator_clean"
if "analyte" in df.columns and "indicator_clean" not in df.columns:
    rename_map["analyte"] = "indicator_clean"
if "result" in df.columns and "result_value" not in df.columns:
    rename_map["result"] = "result_value"

df = df.rename(columns=rename_map)

required_base_cols = ["sample_date", "indicator_clean", "result_value"]
missing_base = [c for c in required_base_cols if c not in df.columns]
if missing_base:
    raise ValueError(f"Missing required columns: {missing_base}. Available columns: {df.columns.tolist()}")

df["sample_date"] = pd.to_datetime(df["sample_date"], errors="coerce")
df["indicator_clean"] = df["indicator_clean"].astype(str).str.strip()

df["indicator_clean"] = df["indicator_clean"].replace({
    "e coli": "E. coli",
    "ecoli": "E. coli",
    "e_coli": "E. coli",
    "E Coli": "E. coli",
    "Enterococci": "Enterococcus",
    "enterococcus": "Enterococcus",
})

df["result_value"] = pd.to_numeric(df["result_value"], errors="coerce")

print("Date range:", df["sample_date"].min(), "to", df["sample_date"].max())
print("Indicators:")
display(df["indicator_clean"].value_counts(dropna=False))

In [ ]:
# ============================================================
# 4. Configuration: thresholds and current best models
# ============================================================

TARGET_CONFIGS = {
    "E. coli": {
        "threshold": 235.0,
        "best_model": "Logistic Regression",
        "feature_set": "expanded_no_tides",
        "binary_threshold": None,
        "target_recall_min": 0.85,
    },
    "Enterococcus": {
        "threshold": 130.0,
        "best_model": "Logistic Regression",
        "feature_set": "base_no_sso",
        "binary_threshold": 0.630,
        "target_recall_min": 0.60,
    },
}

BASE_NO_SSO_FEATURES = [
    "rain_1day",
    "rain_3day_sum",
    "temp_3day_avg",
    "season_wet",
    "adp_days",
    "rain_ratio_1to3",
    "rain_1day_lag1",
    "rain_1day_lag2",
    "first_flush_index",
]

EXPANDED_NO_TIDES_CANDIDATES = BASE_NO_SSO_FEATURES + [
    "rain_7day_sum",
    "rain_14day_sum",
    "rain_3day_max",
    "rain_7day_max",
    "rain_14day_max",
    "rain_3day_avg",
    "rain_7day_avg",
    "rain_14day_avg",
    "rain_3day_rainy_days",
    "rain_7day_rainy_days",
    "rain_14day_rainy_days",
    "rain_intensity_3day",
    "rain_intensity_7day",
    "rain_intensity_14day",
    "temp_7day_avg",
    "temp_14day_avg",
    "temp_3day_min",
    "temp_3day_max",
    "temp_7day_min",
    "temp_7day_max",
    "wet_season_rain_1day",
    "wet_season_rain_3day_sum",
    "wet_season_rain_7day_sum",
    "rain_temp_interaction_1day",
    "rain_temp_interaction_3day",
    "prev_result_value",
    "prev_exceedance",
    "prev_result_to_threshold_ratio",
    "prev3_result_mean",
    "prev5_result_mean",
    "prev3_exceedance_rate",
    "prev5_exceedance_rate",
    "prev3_result_to_threshold_ratio_mean",
    "prev5_result_to_threshold_ratio_mean",
    "days_since_prev_sample",
]

LEAKAGE_COLUMNS = {
    "result_to_threshold_ratio",
    "exceedance",
    "target",
    "binary_target",
    "actual",
}

FEATURE_SETS = {
    "base_no_sso": [c for c in BASE_NO_SSO_FEATURES if c in df.columns and c not in LEAKAGE_COLUMNS],
    "expanded_no_tides": [c for c in EXPANDED_NO_TIDES_CANDIDATES if c in df.columns and c not in LEAKAGE_COLUMNS],
}

for name, features in FEATURE_SETS.items():
    print(name, len(features), "features")
    print(features)

In [ ]:
# ============================================================
# 5. Helper functions (with ChatGPT assistance) 
# ============================================================

def make_target_dataset(source_df, bacteria_name, bacteria_threshold, features):
    subset = source_df[source_df["indicator_clean"] == bacteria_name].copy()
    subset = subset.dropna(subset=["sample_date", "result_value"])
    subset = subset.sort_values("sample_date").reset_index(drop=True)
    subset["actual"] = (subset["result_value"] >= bacteria_threshold).astype(int)

    before = len(subset)
    subset = subset.dropna(subset=features).copy()
    after = len(subset)

    if before != after:
        print(f"{bacteria_name}: dropped {before - after} rows with missing feature values.")

    return subset

def chronological_split(data, train_frac=0.70, val_frac=0.15):
    data = data.sort_values("sample_date").reset_index(drop=True)
    n = len(data)
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))

    train = data.iloc[:train_end].copy()
    val = data.iloc[train_end:val_end].copy()
    test = data.iloc[val_end:].copy()

    return train, val, test

def make_model():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            solver="lbfgs",
            random_state=RANDOM_STATE,
        )),
    ])

def get_probabilities(model, X):
    return model.predict_proba(X)[:, 1]

def metric_row(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

def scan_thresholds(y_true, y_prob, step=0.001):
    rows = []
    for t in np.round(np.arange(0.0, 1.0001, step), 3):
        rows.append(metric_row(y_true, y_prob, t))
    return pd.DataFrame(rows)

def choose_threshold_by_rule(scan_df, min_recall=0.85):
    feasible = scan_df[scan_df["recall"] >= min_recall].copy()
    if feasible.empty:
        chosen = scan_df.sort_values(["recall", "precision", "f1"], ascending=[False, False, False]).iloc[0].copy()
        chosen["selection_rule"] = f"no threshold reached recall >= {min_recall}; chose highest recall"
    else:
        chosen = feasible.sort_values(["precision", "f1", "recall"], ascending=[False, False, False]).iloc[0].copy()
        chosen["selection_rule"] = f"max precision subject to recall >= {min_recall}"
    return chosen

def plot_binary_confusion_matrix(y_true, y_prob, threshold, title):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Safe", "Unsafe"])
    ax.set_yticklabels(["Safe", "Unsafe"])

    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=14, fontweight="bold")

    plt.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# 6. Train current best models (ChatGPT assisted for hyperparameter tuning)
# ============================================================

trained_best = {}
summary_rows = {}
threshold_scans = {}

for bacteria_name, cfg in TARGET_CONFIGS.items():
    feature_set_name = cfg["feature_set"]
    features = FEATURE_SETS[feature_set_name]

    if not features:
        raise ValueError(f"No features found for {bacteria_name} feature set {feature_set_name}")

    print("\n" + "=" * 80)
    print(f"Training current best model for {bacteria_name}")
    print(f"Model: {cfg['best_model']}")
    print(f"Feature set: {feature_set_name}")
    print(f"Features used: {len(features)}")
    print("=" * 80)

    data = make_target_dataset(df, bacteria_name, cfg["threshold"], features)
    train, val, test = chronological_split(data)

    X_train = train[features]
    y_train = train["actual"].astype(int)
    X_val = val[features]
    y_val = val["actual"].astype(int)
    X_test = test[features]
    y_test = test["actual"].astype(int)

    model = make_model()
    model.fit(X_train, y_train)

    val_prob = get_probabilities(model, X_val)
    test_prob = get_probabilities(model, X_test)

    val_scan = scan_thresholds(y_val.values, val_prob, step=0.001)
    chosen = choose_threshold_by_rule(val_scan, min_recall=cfg["target_recall_min"])

    chosen_threshold = float(chosen["threshold"])
    fixed_threshold = cfg.get("binary_threshold", None)

    if fixed_threshold is not None:
        print(f"Known current threshold from prior tuning: {fixed_threshold:.3f}")
        print(f"Validation-selected threshold from this notebook: {chosen_threshold:.3f}")
        final_threshold = float(fixed_threshold)
    else:
        final_threshold = chosen_threshold

    test_metrics = metric_row(y_test.values, test_prob, final_threshold)
    try:
        pr_auc = average_precision_score(y_test.values, test_prob)
    except Exception:
        pr_auc = np.nan
    try:
        roc_auc = roc_auc_score(y_test.values, test_prob)
    except Exception:
        roc_auc = np.nan

    summary_rows[bacteria_name] = {
        "bacteria": bacteria_name,
        "model": cfg["best_model"],
        "feature_set": feature_set_name,
        "threshold": final_threshold,
        "train_rows": len(train),
        "validation_rows": len(val),
        "test_rows": len(test),
        "test_start": test["sample_date"].min(),
        "test_end": test["sample_date"].max(),
        **test_metrics,
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
    }

    trained_best[bacteria_name] = {
        "model": model,
        "features": features,
        "feature_set": feature_set_name,
        "threshold": final_threshold,
        "train": train,
        "val": val,
        "test": test,
        "val_prob": val_prob,
        "test_prob": test_prob,
        "val_scan": val_scan,
    }
    threshold_scans[bacteria_name] = val_scan

best_model_summary = pd.DataFrame(list(summary_rows.values()))
display(best_model_summary)

In [ ]:
# ============================================================
# 7. Binary 2x2 matrices for current best models
# ============================================================

for bacteria_name, obj in trained_best.items():
    y_test = obj["test"]["actual"].astype(int).values
    test_prob = obj["test_prob"]
    threshold = obj["threshold"]

    row = metric_row(y_test, test_prob, threshold)

    title = (
        f"{bacteria_name} Current Best Binary Matrix\n"
        f"threshold={threshold:.3f}, recall={row['recall']:.3f}, precision={row['precision']:.3f}"
    )

    plot_binary_confusion_matrix(y_test, test_prob, threshold, title)

    print(bacteria_name)
    print("Threshold:", threshold)
    print("Accuracy:", round(row["accuracy"], 3))
    print("Precision:", round(row["precision"], 3))
    print("Recall:", round(row["recall"], 3))
    print("F1:", round(row["f1"], 3))
    print("TN, FP, FN, TP:", row["tn"], row["fp"], row["fn"], row["tp"])

In [ ]:
# ============================================================
# 8. Threshold tradeoff tables
# ============================================================

for bacteria_name, obj in trained_best.items():
    y_test = obj["test"]["actual"].astype(int).values
    test_prob = obj["test_prob"]

    test_scan = scan_thresholds(y_test, test_prob, step=0.001)
    print("\n" + "=" * 80)
    print(f"{bacteria_name}: best thresholds subject to recall floors")
    print("=" * 80)

    rows = []
    for recall_floor in [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]:
        feasible = test_scan[test_scan["recall"] >= recall_floor].copy()
        if feasible.empty:
            continue
        best = feasible.sort_values(["precision", "f1", "recall"], ascending=[False, False, False]).iloc[0]
        rows.append({
            "recall_floor": recall_floor,
            "threshold": best["threshold"],
            "precision": best["precision"],
            "recall": best["recall"],
            "f1": best["f1"],
            "tn": best["tn"],
            "fp": best["fp"],
            "fn": best["fn"],
            "tp": best["tp"],
        })

    display(pd.DataFrame(rows))

In [ ]:
# ============================================================
# 9. Optional app Safe / Caution / Unsafe thresholds (ChatGPT assisted) 
# ============================================================

APP_THRESHOLDS = {
    "E. coli": {"safe_max": 0.10, "unsafe_min": 0.50},
    "Enterococcus": {"safe_max": 0.40, "unsafe_min": 0.85},
}

APP_CATEGORY_ORDER = ["Safe", "Caution", "Unsafe"]

def predicted_3level_from_probability(bacteria_name, probability):
    safe_max = APP_THRESHOLDS[bacteria_name]["safe_max"]
    unsafe_min = APP_THRESHOLDS[bacteria_name]["unsafe_min"]

    if probability < safe_max:
        return "Safe"
    elif probability < unsafe_min:
        return "Caution"
    else:
        return "Unsafe"

def actual_3level_from_result(bacteria_name, result_value):
    threshold = TARGET_CONFIGS[bacteria_name]["threshold"]
    if result_value >= threshold:
        return "Unsafe"
    elif result_value >= 0.5 * threshold:
        return "Caution"
    else:
        return "Safe"

app_rows = []
for bacteria_name, obj in trained_best.items():
    test = obj["test"].copy()
    prob = obj["test_prob"]

    test["probability"] = prob
    test["actual_3level"] = test["result_value"].apply(lambda x: actual_3level_from_result(bacteria_name, x))
    test["predicted_3level"] = test["probability"].apply(lambda p: predicted_3level_from_probability(bacteria_name, p))

    matrix = pd.crosstab(
        pd.Categorical(test["actual_3level"], categories=APP_CATEGORY_ORDER, ordered=True),
        pd.Categorical(test["predicted_3level"], categories=APP_CATEGORY_ORDER, ordered=True),
        dropna=False,
    ).reindex(index=APP_CATEGORY_ORDER, columns=APP_CATEGORY_ORDER, fill_value=0)

    print("\n" + bacteria_name + " 3-level app matrix")
    display(matrix)

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(matrix.values)
    ax.set_title(f"{bacteria_name} Safe/Caution/Unsafe Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(APP_CATEGORY_ORDER)
    ax.set_yticklabels(APP_CATEGORY_ORDER)

    for i in range(3):
        for j in range(3):
            ax.text(j, i, str(int(matrix.iloc[i, j])), ha="center", va="center", fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    app_rows.append(test.assign(bacteria=bacteria_name))

app_predictions_current_best = pd.concat(app_rows, ignore_index=True)
display(app_predictions_current_best.head())

In [ ]:
# ============================================================
# 10. Export final AI/model files, outputs, and downloadable ZIP
# ============================================================

from pathlib import Path
import shutil
import json
import joblib

# Saves into the same Drive folder as the CSV.
DRIVE_DIR = Path("/content/drive/MyDrive/Datasets")
FINAL_DIR = DRIVE_DIR / "AquaCast_Model_Pipeline_Version_10_Outputs"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print("Saving final outputs to:")
print(FINAL_DIR)

# ------------------------------------------------------------
# 1. Save trained model files
# ------------------------------------------------------------

if "trained_best" not in globals():
    raise RuntimeError("trained_best was not found. Run the training cells before this export cell.")

model_manifest = {}

for bacteria_name, obj in trained_best.items():
    safe_name = bacteria_name.replace(".", "").replace(" ", "_").lower()

    model_path = FINAL_DIR / f"{safe_name}_current_best_model.joblib"
    info_path = FINAL_DIR / f"{safe_name}_model_info.json"
    features_path = FINAL_DIR / f"{safe_name}_features.txt"

    # Save the fitted sklearn Pipeline.
    joblib.dump(obj["model"], model_path)

    # Save model information needed by frontend/backend later.
    model_info = {
        "bacteria": bacteria_name,
        "model_type": "Logistic Regression sklearn Pipeline with StandardScaler",
        "feature_set": obj["feature_set"],
        "features": obj["features"],
        "binary_threshold": float(obj["threshold"]),
        "app_thresholds": APP_THRESHOLDS.get(bacteria_name, None) if "APP_THRESHOLDS" in globals() else None,
        "bacteria_exceedance_threshold_mpn_per_100ml": float(TARGET_CONFIGS[bacteria_name]["threshold"]),
        "dataset_path": DATA_PATH,
    }

    with open(info_path, "w") as f:
        json.dump(model_info, f, indent=2)

    with open(features_path, "w") as f:
        for feature in obj["features"]:
            f.write(feature + "\n")

    model_manifest[bacteria_name] = {
        "model_file": model_path.name,
        "info_file": info_path.name,
        "features_file": features_path.name,
        "binary_threshold": float(obj["threshold"]),
        "feature_set": obj["feature_set"],
        "n_features": len(obj["features"]),
    }

# Save one combined manifest.
with open(FINAL_DIR / "model_manifest.json", "w") as f:
    json.dump(model_manifest, f, indent=2)

# Save a combined model bundle too.
joblib.dump(trained_best, FINAL_DIR / "aquacast_current_best_models_bundle.joblib")

print("\nSaved trained model files:")
for p in sorted(FINAL_DIR.glob("*.joblib")):
    print(" -", p.name)

# ------------------------------------------------------------
# 2. Save metrics and prediction outputs
# ------------------------------------------------------------

if "best_model_summary" in globals():
    best_model_summary.to_csv(FINAL_DIR / "current_best_model_binary_metrics.csv", index=False)

if "app_predictions_current_best" in globals():
    app_predictions_current_best.to_csv(FINAL_DIR / "current_best_app_predictions.csv", index=False)

# Save test probabilities separately for easier checking.
probability_rows = []
for bacteria_name, obj in trained_best.items():
    test = obj["test"].copy()
    test["bacteria"] = bacteria_name
    test["predicted_probability"] = obj["test_prob"]
    test["binary_threshold"] = obj["threshold"]
    test["predicted_binary"] = (test["predicted_probability"] >= obj["threshold"]).astype(int)
    probability_rows.append(test)

test_probabilities = pd.concat(probability_rows, ignore_index=True)
test_probabilities.to_csv(FINAL_DIR / "current_best_test_probabilities.csv", index=False)

# Save validation threshold scans.
for bacteria_name, obj in trained_best.items():
    safe_name = bacteria_name.replace(".", "").replace(" ", "_").lower()
    obj["val_scan"].to_csv(FINAL_DIR / f"{safe_name}_validation_threshold_scan.csv", index=False)

# ------------------------------------------------------------
# 3. Save confusion-matrix images
# ------------------------------------------------------------

for bacteria_name, obj in trained_best.items():
    safe_name = bacteria_name.replace(".", "").replace(" ", "_").lower()

    y_test = obj["test"]["actual"].astype(int).values
    test_prob = obj["test_prob"]
    threshold = obj["threshold"]
    y_pred = (test_prob >= threshold).astype(int)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(cm)
    ax.set_title(f"{bacteria_name} Binary Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Safe", "Unsafe"])
    ax.set_yticklabels(["Safe", "Unsafe"])

    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=14, fontweight="bold")

    plt.tight_layout()
    fig.savefig(FINAL_DIR / f"{safe_name}_binary_confusion_matrix.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# ------------------------------------------------------------
# 4. Save a README so the files are understandable later
# ------------------------------------------------------------

readme_text = f"""
# AquaCast Model Pipeline Version 10 Current Best Model Outputs

This folder contains the Version 10 final trained AI/model files and outputs for AquaCast / BeachGuard.

Dataset path used:
{DATA_PATH}

Current best models:
- E. coli: Logistic Regression, expanded no-tides feature set
- Enterococcus: Logistic Regression, base no-SSO feature set

Important files:
- ecoli_current_best_model.joblib: trained E. coli sklearn model
- enterococcus_current_best_model.joblib: trained Enterococcus sklearn model
- aquacast_current_best_models_bundle.joblib: combined trained model bundle
- model_manifest.json: feature sets, thresholds, and model file names
- current_best_model_binary_metrics.csv: binary test metrics
- current_best_app_predictions.csv: Safe / Caution / Unsafe app output table
- current_best_test_probabilities.csv: test-set probabilities and binary predictions
- *_validation_threshold_scan.csv: threshold tradeoff scans
- *_binary_confusion_matrix.png: saved confusion-matrix images

Binary bacteria thresholds:
- E. coli unsafe/exceedance: result >= 235 MPN/100 mL
- Enterococcus unsafe/exceedance: result >= 130 MPN/100 mL

App probability thresholds:
- E. coli: Safe < 0.10, Caution 0.10-0.50, Unsafe >= 0.50
- Enterococcus: Safe < 0.40, Caution 0.40-0.85, Unsafe >= 0.85
"""

with open(FINAL_DIR / "README_AquaCast_Current_Best_Outputs.txt", "w") as f:
    f.write(readme_text.strip() + "\n")

# ------------------------------------------------------------
# 5. Zip everything and download it
# ------------------------------------------------------------

zip_base = DRIVE_DIR / "AquaCast_Model_Pipeline_Version_10_Outputs"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=FINAL_DIR)

print("\nFinal output folder:")
print(FINAL_DIR)

print("\nZip file:")
print(zip_path)

print("\nFiles saved:")
for p in sorted(FINAL_DIR.glob("*")):
    print(" -", p.name)

# Auto-download the ZIP in Colab.
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("\nNot running in Colab, so automatic download was skipped.")